# Demonstration of Robusta-HMF
Just a toy example to see how it works.

## notes:
- Apologies for old-school code!

In [ ]:
!pip install robusta-hmf

In [ ]:
import numpy as np
import pylab as plt
from robusta_hmf import Robusta

In [ ]:
# make fake data
rng = np.random.default_rng(17) # the most random number
N, M, K = 1024, 1536, 16
foo = np.random.normal(size=(21, K)) # brittle
design_matrix = np.concatenate((np.cos(2. * np.pi * np.outer(np.arange(M), np.arange(11)) / M),
                                np.sin(2. * np.pi * np.outer(np.arange(M), np.arange(1, 11)) / M)),
                               axis=1)
true_eigs = (design_matrix @ foo).T
true_coes = np.random.normal(size=(N, K))
sigma = 1.5
outliers = 10. * (rng.uniform(size=(N, M)) > 0.98).astype(float)
data = true_coes @ true_eigs + sigma * rng.normal(size=(N, M)) + outliers
ivar = np.zeros_like(data) + 1. / (sigma * sigma)
print(data.shape)

In [ ]:
# split data into an A sample and a B sample
foo = np.arange(N)
rng.shuffle(foo)
Aindx = foo[:N // 2]
Bindx = foo[N // 2:]

In [ ]:
# plot some of the data from sample B just to look
f = plt.figure(figsize=(15, 15))
offset = 100
for i in range(16):
    f.gca().plot(data[Bindx[i]] + i * offset, color="k")

In [ ]:
# train two Robusta models
modelA = Robusta(rank=K, robust=True)
modelB = Robusta(rank=K, robust=True)
fooA = modelA.fit(data[Aindx], ivar[Aindx])
fooB = modelB.fit(data[Bindx], ivar[Bindx])

In [ ]:
# plot the eigenspectra
eigsA = modelA.basis_vectors().T
eigsB = modelB.basis_vectors().T
f = plt.figure(figsize=(15, 15))
offset = 0.2
for i in range(K):
    f.gca().plot(eigsA[i] + i * offset)
    f.gca().plot(eigsB[i] + i * offset, alpha=0.5)

In [ ]:
# infer at the other (complementary, or held-out) samples
foo, _ = modelA.infer(data[Bindx], ivar[Bindx])
print(foo)
bar = modelA.synthesize(foo)
print(bar.shape)

In [ ]:
# plot examples
f = plt.figure(figsize=(15, 15))
offset = 100
for i in range(16):
    f.gca().plot(data[Bindx[i]] + i * offset, color="k")
    f.gca().plot(bar[i] + i * offset, color="r")